#same thing write
Determining the Optimal Number of Hidden Layers and Neurons for an Artificial Neural Network (ANN)

###This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you make an informed decision:
###- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
###- Grid Search/Random Search: Use Grid Search or Random Search to try different architectures.
### - Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
### - Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
###  - The number of neurons in the hidden layer should generally be between the size of the input layer and the size of the output layer.
###   - A common practice is to start with 1–2 hidden layers.

In [40]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle


In [41]:
data = pd.read_csv('Churn_Modelling.csv')

data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [42]:
## (create keras model) Define a function to create the model and try different parameters (KerasClassifier)

def create_model(neurons=32, layers=1):
    model = Sequential()

    model.add(
        Dense(
            neurons,
            activation='relu',
            input_shape=(X_train.shape[1],)
        )
    )

    # additional hidden layer 

    for _ in range(layers - 1):
        model.add(Dense(neurons, activation='relu'))

    # output layer

    model.add(Dense(1, activation='sigmoid'))

    # compile model 
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [43]:
## create a KerasClassifier for use in GridSearchCV
model = KerasClassifier(
    layers = 1,
    neurons = 32,
    build_fn=create_model,
    verbose=0
)

In [44]:
# define the grid search parameters
param_grid = {
    # Parameters of create_model()
    'model__neurons': [16, 32, 64, 128],
    'model__layers': [1, 2],

      # Training parameters
    'epochs': [50, 100],

}

In [47]:
## Perform grid search 
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    verbose = 1
)
#TRAIN GRID SEARCH
grid_result = grid.fit(X_train, y_train)

#print the best parameters and score
print("\nBest Parameters:")
print(grid_result.best_params_)

print("\nBest Cross Validation Accuracy:")
print(grid_result.best_score_)

Fitting 3 folds for each of 16 candidates, totalling 48 fits


c:\Users\anmol\OneDrive\Desktop\ANN Classification\venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\anmol\OneDrive\Desktop\ANN Classification\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Best Parameters:
{'epochs': 50, 'model__layers': 2, 'model__neurons': 16}

Best Cross Validation Accuracy:
0.8574991516908099
